# Preprocessing
Feature selection, cleaning, encoding, scaling, train/test split.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

df = pd.read_parquet('../data/loans_filtered.parquet')
df = df.copy()  # defragment
print('Loaded shape:', df.shape)

Loaded shape: (1348092, 152)


## Step 1: Feature Selection
Keep only features relevant to our hypotheses + known credit risk predictors.

In [2]:
FEATURES = [
    # Hypothesis features
    'dti',                  # H1
    'earliest_cr_line',     # H2 - will engineer
    'purpose',              # H3
    'annual_inc',           # H4
    'pub_rec',              # H5

    # Strong credit risk predictors
    'loan_amnt',
    'int_rate',
    'installment',
    'grade',
    'sub_grade',
    'emp_length',
    'home_ownership',
    'verification_status',
    'open_acc',
    'revol_bal',
    'revol_util',
    'total_acc',
    'mort_acc',
    'delinq_2yrs',
    'inq_last_6mths',

    # Target
    'target'
]

df = df[FEATURES].copy()
print('Shape after feature selection:', df.shape)
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Shape after feature selection: (1348092, 21)

Missing values:
dti                      407
earliest_cr_line          62
purpose                   33
annual_inc                37
pub_rec                   62
loan_amnt                 33
int_rate                  33
installment               33
grade                     33
sub_grade                 33
emp_length             78578
home_ownership            33
verification_status       33
open_acc                  62
revol_bal                 33
revol_util               930
total_acc                 62
mort_acc               50063
delinq_2yrs               62
inq_last_6mths            63
target                    33
dtype: int64


## Step 2: Feature Engineering

In [3]:
# H2: Convert earliest_cr_line to credit_history_years
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y', errors='coerce')
df['credit_history_years'] = (pd.Timestamp('2018-12-31') - df['earliest_cr_line']).dt.days / 365
df = df.drop(columns=['earliest_cr_line'])

# H5: Convert pub_rec to binary flag
df['has_derogatory'] = (df['pub_rec'] > 0).astype(int)
df = df.drop(columns=['pub_rec'])

# Cap annual_inc at 99th percentile
income_cap = df['annual_inc'].quantile(0.99)
df['annual_inc'] = df['annual_inc'].clip(0, income_cap)

# emp_length: convert to numeric
emp_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3,
    '4 years': 4, '5 years': 5, '6 years': 6, '7 years': 7,
    '8 years': 8, '9 years': 9, '10+ years': 10
}
df['emp_length'] = df['emp_length'].map(emp_map)

# grade: convert to numeric (A=1, B=2, ..., G=7)
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
df['grade'] = df['grade'].map(grade_map)

print('Shape after engineering:', df.shape)
print('\nNew features added: credit_history_years, has_derogatory')
print('Engineered: emp_length, grade')

Shape after engineering: (1348092, 21)

New features added: credit_history_years, has_derogatory
Engineered: emp_length, grade


## Step 3: Handle Missing Values

In [4]:
print('Missing before imputation:')
missing = df.isnull().sum()
print(missing[missing > 0])

# Numeric: fill with median
num_cols = df.select_dtypes(include='number').columns.tolist()
num_cols = [c for c in num_cols if c != 'target']
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Categorical: fill with mode
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print('\nMissing after imputation:')
print(df.isnull().sum().sum(), 'nulls remaining')

Missing before imputation:
dti                       407
purpose                    33
annual_inc                 37
loan_amnt                  33
int_rate                   33
installment                33
grade                      33
sub_grade                  33
emp_length              78578
home_ownership             33
verification_status        33
open_acc                   62
revol_bal                  33
revol_util                930
total_acc                  62
mort_acc                50063
delinq_2yrs                62
inq_last_6mths             63
target                     33
credit_history_years       62
dtype: int64

Missing after imputation:
33 nulls remaining


## Step 4: Encode Categorical Features

In [5]:
# Drop sub_grade (redundant with grade)
df = df.drop(columns=['sub_grade'])

# One-hot encode low-cardinality categoricals
cat_cols = df.select_dtypes(include='object').columns.tolist()
print('Categorical columns to encode:', cat_cols)

df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print('Shape after encoding:', df.shape)

Categorical columns to encode: ['purpose', 'home_ownership', 'verification_status']
Shape after encoding: (1348092, 37)


## Step 5: Train/Test Split

In [7]:
X = df.drop(columns=['target'])
y = df['target']

# Drop rows where target is NaN
mask = y.notna()
X = X[mask]
y = y[mask]

print('Rows dropped due to NaN target:', (~mask).sum())

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Train shape:', X_train.shape)
print('Test shape: ', X_test.shape)
print('\nTrain target split:')
print(y_train.value_counts(normalize=True).mul(100).round(2))
print('\nTest target split:')
print(y_test.value_counts(normalize=True).mul(100).round(2))

Rows dropped due to NaN target: 33
Train shape: (1078447, 36)
Test shape:  (269612, 36)

Train target split:
target
0.0    80.02
1.0    19.98
Name: proportion, dtype: float64

Test target split:
target
0.0    80.02
1.0    19.98
Name: proportion, dtype: float64


## Step 6: Scale Numeric Features

In [8]:
scaler = StandardScaler()

# Only scale numeric columns (not binary/dummy columns)
scale_cols = [c for c in X_train.columns if X_train[c].nunique() > 2]

X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])  # use train stats only

print('Scaled', len(scale_cols), 'numeric columns')
print('Sample scaled values (X_train first row):')
print(X_train[scale_cols[:5]].iloc[0].round(3))

Scaled 15 numeric columns
Sample scaled values (X_train first row):
dti            1.288
annual_inc    -1.269
loan_amnt     -1.401
int_rate       0.173
installment   -1.386
Name: 378278, dtype: float64


## Step 7: Save Processed Data

In [9]:
import pickle

X_train.to_parquet('../data/X_train.parquet', index=False)
X_test.to_parquet('../data/X_test.parquet', index=False)
y_train.to_frame().to_parquet('../data/y_train.parquet', index=False)
y_test.to_frame().to_parquet('../data/y_test.parquet', index=False)

# Save scaler for use in production
with open('../data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Saved:')
print('  X_train.parquet:', X_train.shape)
print('  X_test.parquet: ', X_test.shape)
print('  y_train.parquet:', y_train.shape)
print('  y_test.parquet: ', y_test.shape)
print('  scaler.pkl')
print('\nFeature count:', X_train.shape[1])
print('\nAll features:')
print(list(X_train.columns))

Saved:
  X_train.parquet: (1078447, 36)
  X_test.parquet:  (269612, 36)
  y_train.parquet: (1078447,)
  y_test.parquet:  (269612,)
  scaler.pkl

Feature count: 36

All features:
['dti', 'annual_inc', 'loan_amnt', 'int_rate', 'installment', 'grade', 'emp_length', 'open_acc', 'revol_bal', 'revol_util', 'total_acc', 'mort_acc', 'delinq_2yrs', 'inq_last_6mths', 'credit_history_years', 'has_derogatory', 'purpose_credit_card', 'purpose_debt_consolidation', 'purpose_educational', 'purpose_home_improvement', 'purpose_house', 'purpose_major_purchase', 'purpose_medical', 'purpose_moving', 'purpose_other', 'purpose_renewable_energy', 'purpose_small_business', 'purpose_vacation', 'purpose_wedding', 'home_ownership_MORTGAGE', 'home_ownership_NONE', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'verification_status_Source Verified', 'verification_status_Verified']


## Feature Significance Insights (from 03b_feature_significance.ipynb)

### Tests Run
- Point-Biserial Correlation, Mann-Whitney U, Chi-Square, Correlation Matrix, VIF

### Key Findings

**All 19 features are statistically significant (p = 0.0)**
With 1.3M rows, even tiny differences become significant. Statistical significance alone is not enough — we also need VIF and correlation checks.

### Multicollinearity Problems Found

| Pair | Correlation | Action |
|---|---|---|
|  vs  | 0.952 | Drop  — grade is more interpretable |
|  vs  | 0.953 | Drop  — loan_amnt is the raw signal |
|  vs  | 0.701 | Drop  — open accounts are more current |

### VIF Verdicts

| Feature | VIF | Verdict |
|---|---|---|
| int_rate | 75.78 | DROP (severe) |
| grade | 53.52 | DROP (severe) — kept because more interpretable than int_rate |
| loan_amnt | 43.99 | DROP (severe) — kept as primary loan size signal |
| installment | 43.28 | DROP (severe) |
| total_acc | 13.17 | DROP (severe) |
| open_acc | 11.70 | DROP (severe) — kept over total_acc |

### Final Decision: 3 Features Dropped for Modeling



**Final feature count going into 04_modeling.ipynb: 33 features**
(36 from this notebook minus 3 redundant features)
